# Option 2 finish-up run

Run every cell top to bottom, in order. Put your normal setup cell (clone/pin + `bind()` to Drive) in the empty cell right below this one, then run it, then continue.

GPU cells: 2-6. Judge cell: 7. Everything from cell 8 on is CPU-only (fine to keep running here, or move to a CPU runtime / your PC once results/ is synced).

In [ ]:
# <-- put your usual setup cell here (clone/pin repo, bind() to Drive) and run it first -->


## Cell 1 — pull latest code + check current state before spending GPU time

In [ ]:
!git pull origin agent/c-quadrant-end-to-end-e0e2317a
!python -m src.analysis.v2_pipeline status


## Cell 2 — re-extract the 5 stale stages against the current 654-row benchmark
M0, M1, M1_alt, M3_direct, M3_direct_alt were still built against the old 370-row benchmark; M2, M2_alt, M3, M3_alt are already fresh.

In [ ]:
!python -m src.analysis.v2_pipeline extract --stages M0 M1 M1_alt M3_direct M3_direct_alt
!python -m src.analysis.verify_activations


## Cell 3 — refresh behavioral responses for those same 5 stages

In [ ]:
!python -m src.analysis.v2_pipeline behavior --stages M0 M1 M1_alt M3_direct M3_direct_alt


## Cell 4 — refresh directions + probes for ALL 9 stages
`--force` matters: without it, `direction` silently no-ops because stale direction files already exist on disk.

In [ ]:
!python -m src.analysis.v2_pipeline direction --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt --force
!python -m src.analysis.v2_pipeline probes    --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt


## Cell 5 — causal ablation for the 3 missing branches
M3 is already done (CF1/CF2) — don't redo it.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3_direct --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_alt --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_direct_alt --conditions baseline ablated_AD ablated_random


## Cell 6 — steering for the 2 missing branches
M3 and M3_alt's dose-response (6 cells) are already done.

In [ ]:
!python -m src.analysis.v2_pipeline steering --stage M3_direct --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_direct_alt --alpha-coefficients 0.5 1.0 2.0


## Cell 7 — rebuild the consolidated manifest and re-run the judge (LAST GPU STEP)
`--from-results-dir results` rescans results/ and rebuilds the manifest, so it automatically picks up the 3 new causal branches' response files. This is the long one — see the chat estimate before running.

In [ ]:
!python -m src.analysis.behavioral_judges \
  --response-manifest results/manifests/consolidated_judge.json \
  --from-results-dir results \
  --out-dir results/behavioral_judges_v2 \
  --run-live --scope confirmatory


---
# Everything below is CPU-only
Fine to keep running here, or switch to a CPU runtime, or run locally once `results/` is synced to your PC.

## Cell 8 — confirmatory endpoints (CF1 + per-branch CF2), auto-finds the newest judge output

In [ ]:
import glob
judged_files = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))
assert judged_files, 'No judge output found - did Cell 7 finish?'
latest_judged = judged_files[-1]
print('Using judge file:', latest_judged)
!python -m src.analysis.confirmatory_behavioral_endpoints \
  --judged {latest_judged} \
  --benchmark data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl \
  --out results/summaries/confirmatory_endpoints.json


## Cell 9 — the full descriptive / geometric / decodability story (all 9 stages, all branches)

In [ ]:
!python -m src.analysis.subspace_geometry
!python -m src.analysis.projection_trajectory
!python -m src.analysis.direction_decodability
!python -m src.analysis.representation_robustness
!python -m src.interpretability.bottleneck_layer
!python -m src.interpretability.bootstrap_direction_stability
!python -m src.interpretability.bootstrap_cross_branch_difference
!python -m src.interpretability.paired_deep_layer_stability_test --seed 20260904
!python -m src.analysis.summarize_probe_findings
!python -m src.analysis.summarize_cross_branch


## Cell 10 — descriptive causal/steering summaries for the new files

In [ ]:
import glob
for stage in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    f = f'results/raw/causal_ablation_v2_{stage}_L24-28.json'
    get_ipython().system(f'python -m src.analysis.summarize_causal_ablation --file {f}')
    get_ipython().system(f'python -m src.analysis.mcnemar_causal_ablation --file {f} --conditions {stage}_baseline {stage}_ablated_AD')
    get_ipython().system(f'python -m src.analysis.bootstrap_causal_effect --file {f} --quadrant A --category refusal')

for f in glob.glob('results/raw/steering_v2_M3_direct*_QABCD.json'):
    get_ipython().system(f'python -m src.analysis.summarize_steering --file {f}')


## Done
Send back (or paste) `results/summaries/confirmatory_endpoints.json` and the printed output of cells 9-10 for interpretation.